# Class imbalance and cost-sensitive learning

Run from top to bottom. Quick mode is a patient subset, not final results. Notebook outputs are separate from production reports.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'features.py').exists()), None)
assert ROOT is not None, 'Open the notebook from the project repository'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.notebook_utils import *
QUICK = True  # False uses the entire eligible dataset; keep consistent across notebooks
OUT = ROOT/'reports'/('notebook_quick' if QUICK else 'notebook_full')
OUT.mkdir(parents=True, exist_ok=True)


Load training and validation

In [ ]:
raw, x, y, g, parts = load_cohort(ROOT, QUICK)
train, cal, tune, test = (parts[k] for k in ("train","calibration","threshold","test"))
X, Y, G = x.iloc[train], y.iloc[train], g.iloc[train]
display(split_table(y,g,parts))

Fit XGBoost with different imbalance treatments. ADASYN interpolates encoded columns; SMOTENC preserves categorical types.

In [ ]:
from src.models.boosting import models
from src.models.imbalance import weights
rows=[]; predictions={}
for kind in ['none','weighted','SMOTENC','ADASYN']:
    m=make_model(X,models()['XGBoost'],kind=kind,quick=QUICK)
    kw={'model__sample_weight':weights(Y)} if kind=='weighted' else {}
    m.fit(X,Y,**kw); p=m.predict_proba(x.iloc[tune])[:,1]; predictions[kind]=p
    rows.append({'model':kind,**scores(y.iloc[tune],p)})
r=pd.DataFrame(rows); display(r); r.to_csv(OUT/'imbalance.csv',index=False)

Compare recall, precision and average precision at the default threshold

In [ ]:
r.set_index('model')[['recall','precision','pr_auc']].plot.bar(); plt.tight_layout(); plt.show(); plot_curves(y.iloc[tune],predictions)